# Project V: Kaggle evaluation session
Runs a list of `projectv` commands (held-out sweeps, analyses) as parallel subprocesses on
checkpoints and results attached as inputs. Outputs go to `/kaggle/working/results`.

In [ ]:
# PARAMETERS (generated by scripts/make_kaggle_kernels.py --commands)
COMMANDS = ["python -m pytest -q tests/test_envs.py"]
SHA = "main"

In [ ]:
import glob, os, shutil, subprocess, time
REPO = "https://github.com/Darsh-14/Project-v.git"
SRC, WORK = "/kaggle/tmp/pv", "/kaggle/working"
RES, LOGS = f"{WORK}/results", f"{WORK}/logs"
os.makedirs(LOGS, exist_ok=True)
def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)
sh(f"rm -rf {SRC} && git clone -q {REPO} {SRC} && cd {SRC} && git checkout -q {SHA} && git log -1 --oneline")
sh(f"cd {SRC} && pip install -q -r requirements.lock && pip install -q -e . --no-deps")
sh("python --version; nproc; free -g | head -2")

In [ ]:
# Inputs: results/ trees of attached kernels (any mount depth), merged into results/.
for prev in sorted({os.path.dirname(c) for c in glob.glob("/kaggle/input/**/results/ckpt", recursive=True)}):
    print("input:", prev)
    shutil.copytree(prev, RES, dirs_exist_ok=True)
# the repo's committed results (frozen grids, hashes, eval CSVs) take part as well
shutil.copytree(f"{SRC}/results", RES, dirs_exist_ok=True)
os.makedirs(f"{SRC}/results", exist_ok=True)
sh(f"rm -rf {SRC}/results && ln -s {RES} {SRC}/results && ls {RES}")

In [ ]:
env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
procs = []
for i, cmd in enumerate(COMMANDS):
    log = open(f"{LOGS}/cmd{i:02d}.log", "a")
    procs.append((cmd, subprocess.Popen(cmd, shell=True, cwd=SRC, env=env, stdout=log, stderr=subprocess.STDOUT)))
    print("launched", i, cmd, flush=True)
t0 = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(300)
    print(f"--- {(time.time() - t0) / 3600:.2f} h", [p.poll() for _, p in procs], flush=True)
for i, (cmd, p) in enumerate(procs):
    print(i, "rc", p.returncode, cmd)
    print("".join(open(f"{LOGS}/cmd{i:02d}.log").readlines()[-4:]))

In [ ]:
# keep outputs small: drop copied checkpoints/replay buffers from the output
shutil.rmtree(f"{RES}/ckpt", ignore_errors=True)
sh(f"du -sh {RES}/* || true")